# Bitino quickstart: a 1-bit ImageNet classifier in a few lines

[Bitino](https://github.com/fedeloper/bitino) runs **binary neural networks** (1-bit weights and activations:
XNOR + popcount instead of multiply-adds) on CPUs: ARM NEON on phones, AVX2 on x86-64.

This notebook installs the Python package (the native runtime is inside the wheel), downloads a converted
ImageNet model, classifies an image, checks the runtime against the reference numpy executor and times it.

**About the timings below:** Colab runs this notebook on a **cloud x86-64 server CPU** (a shared virtual core,
not a phone). The numbers measure that machine only. The README's phone numbers were measured on the phones
themselves; they are shown at the end for comparison, not reproduced here.

In [ ]:
!pip install -q "bitino[image] @ git+https://github.com/fedeloper/bitino"   # builds the native runtime (~2 min on Colab)

In [ ]:
import os, platform, subprocess
import numpy as np
import bitino

print("bitino", bitino.__version__, "| runtime", bitino.version())
print("CPU features the runtime detected:", bitino.cpu_info())
cpu_model = subprocess.run("grep -m1 'model name' /proc/cpuinfo | cut -d: -f2", shell=True,
                           capture_output=True, text=True).stdout.strip()
print("this machine:", cpu_model or platform.processor(), "|", os.cpu_count(), "logical CPUs (cloud x86 server, not a phone)")

## 1. A redistributable model

Converted files are downloaded only for models whose **weights licence is explicitly permissive** (MIT,
Apache-2.0 or BSD stated for the weights). Every other model of the zoo is converted from the authors' original
checkpoint by its fetch recipe (`bitino fetch <id>` in a clone of the repository); the converted files are not
redistributed.

In [ ]:
from bitino import zoo

redistributable = sorted(m for m in zoo.recipes() if zoo.release_entry(m)[0])
print(len(redistributable), "redistributable models:")
for m in redistributable:
    print("  ", m)

MODEL = "ab-bnn-imagenet-reactnet18"      # ReActNet-18 (AB-BNN), ImageNet, MIT-licensed weights
entry, licence = zoo.release_entry(MODEL)
print("\n", MODEL, "weights licence:", licence)
print("attribution:", entry.get("attribution"))

In [ ]:
m = bitino.load(MODEL)                     # downloads the converted file (SHA-256 checked) into ~/.cache/bitino
print(m)
print("inputs: ", m.inputs)
print("outputs:", m.outputs)
print("preprocessing:", m.preprocess)
print("kernels chosen on this CPU:", m.describe()["kernels"])

## 2. Classify an image

The image is decoded and preprocessed exactly as the model's accuracy check was (here: the torchvision ImageNet
eval transform, re-implemented with Pillow). Replace the URL, or upload your own file in the Colab file panel and set
`IMAGE` to its path.

In [ ]:
import json, urllib.request
from PIL import Image

IMAGE = "grace_hopper.jpg"
if not os.path.exists(IMAGE):
    urllib.request.urlretrieve("https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg", IMAGE)
labels_file = "imagenet_class_index.json"
if not os.path.exists(labels_file):
    urllib.request.urlretrieve("https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json", labels_file)
labels = {int(k): v[1] for k, v in json.load(open(labels_file)).items()}

logits = m.predict_image(IMAGE)[0]
p = np.exp(logits - logits.max()); p /= p.sum()
for k in np.argsort(-logits)[:5]:
    print(f"{labels[k]:>24s}  {100 * p[k]:5.1f} %")
Image.open(IMAGE).resize((224, int(224 * Image.open(IMAGE).height / Image.open(IMAGE).width)))

## 3. Checked, not trusted

Stage `neon` runs the binary layers on the SIMD kernels (AVX2 here, NEON on ARM) with the float layers in a fixed
order; its output must be **bit-identical** to the reference numpy executor running the same file. The default
stage `all` adds float fusion and faster float kernels (equal up to float rounding).

In [ ]:
from bitino import executor, format as bformat

x = bitino.images.load_image(IMAGE, m.preprocess)
g = bformat.read(m.path)
ref = executor.run(g, {g.inputs[0]["name"]: x})[g.outputs[0]].astype(np.float32).reshape(1, -1)

with bitino.load(MODEL, stage="neon") as exact:
    y_neon = exact.predict(x)
print("stage neon == numpy executor, bit for bit:", np.array_equal(y_neon.view(np.uint32), ref.view(np.uint32)))
print("stage all: max |difference| to the reference =", float(np.abs(m.predict(x) - ref).max()))

## 4. Speed on this Colab CPU

Median latency of one inference (batch 1), timing the native run only, after a warm-up. **This is a cloud x86-64
server core, shared with other tenants: expect noise, and do not compare it with a phone.**

In [ ]:
results = {}
for threads in sorted({1, 2, os.cpu_count() or 1}):
    results[threads] = m.benchmark(iterations=50, warmup=10, threads=threads)
    print(f"Colab CPU (cloud x86-64), {threads} thread(s): median {results[threads]:.2f} ms per image")

## 5. For comparison: the README's phone numbers

Measured on the phones with `bitino_bench` (method: `docs/benchmarking.md`; clock state and date per run in
`results/speed_<device>.json`). Each cell: 1 thread pinned to the fastest core / 4 threads, in ms; the second line
is the full-integer int8 file. Markers are explained in the table's legend.

In [ ]:
from IPython.display import Markdown, display

url = "https://raw.githubusercontent.com/fedeloper/bitino/main/docs/SPEED.md"
try:
    speed = urllib.request.urlopen(url, timeout=30).read().decode()
    rows = speed.splitlines()
    head = next(i for i, r in enumerate(rows) if r.startswith("| model |"))
    row = next(r for r in rows if r.startswith(f"| `{MODEL}` |"))
    legend = next((r for r in rows[head:] if r.startswith("Each cell")), "")
    display(Markdown("\n".join([rows[head], rows[head + 1], row]) + "\n\n" + legend))
except Exception as e:
    print("could not fetch the phone table:", e, "\nSee", url)
print(f"This notebook, Colab CPU (cloud x86-64 server): 1 thread {results[1]:.2f} ms")

## Next

* `bitino list --converted` (in a terminal cell: `!bitino list --converted`) lists every converted model with its
  accuracy check and licence; `!bitino info <id>` shows the paper, licences and recipe.
* `bitino.load("model.bitino", stage="int8_full")`, `threads=`, `affinity=`: see `docs/python.md`.
* Run it on a phone: the Android build and the C API are in the repository (`docs/android-deployment.md`).